In [1]:
import pickle
import os
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, auc, accuracy_score)
from scipy.stats import ttest_rel
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout,
                                     Multiply, Reshape, BatchNormalization, GlobalAveragePooling1D,
                                     Lambda)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout, Multiply, Lambda
from tensorflow.keras.layers import BatchNormalization, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
df= pd.read_csv("UNSW_augmented_filtered_data.csv")
df.shape

(257673, 44)

In [5]:
# Select features (exclude 'attack_cat' and 'label')
X = df.drop(['attack_cat', 'label'], axis=1)

# Select target
y = df['attack_cat']

In [6]:
# Encode the labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded = pd.get_dummies(y_encoded).values  # One-hot encode for multiclass classification

In [8]:
# Define CNN Model
def create_cnn_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define LSTM Model
def create_lstm_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = LSTM(64, return_sequences=True)(inputs)
    x = BatchNormalization()(x)
    x = LSTM(32)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define FNN Model
def create_fnn_model(input_shape, num_classes):
    inputs = Input(shape=(input_shape,))
    x = Dense(128, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model

In [9]:
from memory_profiler import memory_usage

In [10]:
def train_ensemble():
    return ensemble_model.fit(
        [X_train_cnn, X_train_cnn, X_train], y_train,
        epochs=10,
        batch_size=128,
        validation_split=0.1,
        verbose=0,
        validation_data=([X_test_cnn, X_test_cnn, X_test], y_test)
    )

In [11]:
# Convert one-hot labels back to class labels for StratifiedKFold
y_labels = np.argmax(y_encoded, axis=1)
# --------------------------------- Part 4: 5-Fold Cross-Validation Setup ---------------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ensemble_accs, cnn_accs, lstm_accs, fnn_accs = [], [], [], []
ensemble_times, cnn_times, lstm_times, fnn_times = [], [], [], []
ensemble_memory_usages, cnn_memory_usages, lstm_memory_usages, fnn_memory_usages = [], [], [], []
all_y_test = []
all_ensemble_pred = []
all_cnn_pred=[]
all_lstm_pred=[]
all_fnn_pred=[]
all_ensemble_prob = []
all_cm_ensemble = []
all_cm_cnn = []
all_cm_lstm = []
all_cm_fnn = []
attention_weights=[]
avg_atts=[]
fold = 1
for train_idx, test_idx in kfold.split(X, y_labels):
    print(f"\n=== Fold {fold} ===")
    fold += 1

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    X_train_cnn = X_train.to_numpy().reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test_cnn = X_test.to_numpy().reshape(X_test.shape[0], X_test.shape[1], 1)

    # Number of classes in the dataset
    num_classes = y_train.shape[1]
    # CNN
    cnn_model = create_cnn_model(X_train_cnn.shape[1:], num_classes)
    cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    cnn_mem_usage, cnn_history = memory_usage(
    (cnn_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True )
    #cnn_history = cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    cnn_times.append(end - start)
    cnn_peak_memory = max(cnn_mem_usage)
    cnn_memory_usages.append(cnn_peak_memory)
    cnn_pred = (cnn_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    cnn_accs.append(accuracy_score(y_test, cnn_pred))
    
    # LSTM
  
    lstm_model = create_lstm_model(X_train_cnn.shape[1:], num_classes)
    lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    lstm_mem_usage, lstm_history = memory_usage(
    (lstm_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
     interval=0.1,retval=True)
    #lstm_history= lstm_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    lstm_times.append(end - start)
    lstm_peak_memory = max(lstm_mem_usage)
    lstm_memory_usages.append(lstm_peak_memory)
    lstm_pred = (lstm_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    lstm_accs.append(accuracy_score(y_test, lstm_pred))
    
     # FNN
    
    fnn_model = create_fnn_model(X_train.shape[1], num_classes)
    fnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    fnn_mem_usage, fnn_history = memory_usage(
    (fnn_model.fit, (X_train, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True)
    #fnn_history=fnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    fnn_times.append(end - start)
    fnn_peak_memory = max(fnn_mem_usage)
    fnn_memory_usages.append(fnn_peak_memory)
    fnn_pred = (fnn_model.predict(X_test,verbose=0) > 0.5).astype(int)
    fnn_accs.append(accuracy_score(y_test, fnn_pred))
    
     # Ensemble
   
    cnn_probs = cnn_model.predict(X_test_cnn,verbose=0)
    lstm_probs = lstm_model.predict(X_test_cnn,verbose=0)
    fnn_probs = fnn_model.predict(X_test,verbose=0)

    # Static average of predicted probabilities
    ensemble_prob = (cnn_probs + lstm_probs + fnn_probs) / 3.0
    ensemble_pred = (ensemble_prob > 0.5).astype(int)

    ensemble_accs.append(accuracy_score(y_test, ensemble_pred))
    ensemble_times.append(cnn_times[-1] + lstm_times[-1] + fnn_times[-1])  # Total time of all models
    ensemble_memory_usages.append(max([cnn_memory_usages[-1], lstm_memory_usages[-1], fnn_memory_usages[-1]]))  # Peak of three
    all_y_test.append(y_test)
    all_ensemble_pred.append(ensemble_pred)
    all_ensemble_prob.append(ensemble_prob)
    


=== Fold 1 ===

=== Fold 2 ===

=== Fold 3 ===

=== Fold 4 ===

=== Fold 5 ===


In [27]:

# --------------------------------- Part 5: Report Results ---------------------------------
def report_scores(name, scores, times, memories):
    print(f"{name}: Accuracy = {np.mean(scores):.4f} ± {np.std(scores):.4f}, "
          f"Time = {np.mean(times):.2f}s ± {np.std(times):.2f}s, "
          f"Memory = {np.mean(memories):.2f} MiB ± {np.std(memories):.2f} MiB")

print("\n=== 5-Fold Cross-validation Results ===")
report_scores("CNN", cnn_accs, cnn_times, cnn_memory_usages)
report_scores("LSTM", lstm_accs, lstm_times, lstm_memory_usages)
report_scores("FNN", fnn_accs, fnn_times, fnn_memory_usages)
report_scores("Ensemble", ensemble_accs, ensemble_times, ensemble_memory_usages)


=== 5-Fold Cross-validation Results ===
CNN: Accuracy = 0.8448 ± 0.0137, Time = 680.92s ± 93.25s, Memory = 744.59 MiB ± 123.73 MiB
LSTM: Accuracy = 0.8264 ± 0.0624, Time = 33046.13s ± 56933.63s, Memory = 803.20 MiB ± 111.82 MiB
FNN: Accuracy = 0.8974 ± 0.0064, Time = 323.38s ± 31.32s, Memory = 815.22 MiB ± 106.29 MiB
Ensemble: Accuracy = 0.8650 ± 0.0216, Time = 34050.42s ± 56997.44s, Memory = 815.22 MiB ± 106.29 MiB


In [28]:

# --------------------------------- Part 6: Statistical Significance Testing ---------------------------------
print("\n=== Paired t-tests ===")
print("Ensemble vs CNN:", ttest_rel(ensemble_accs, cnn_accs))
print("Ensemble vs LSTM:", ttest_rel(ensemble_accs, lstm_accs))
print("Ensemble vs FNN:", ttest_rel(ensemble_accs, fnn_accs))


=== Paired t-tests ===
Ensemble vs CNN: TtestResult(statistic=1.7644445048764066, pvalue=0.15242420442825763, df=4)
Ensemble vs LSTM: TtestResult(statistic=1.8539061479351295, pvalue=0.13736642480059102, df=4)
Ensemble vs FNN: TtestResult(statistic=-2.69406649164727, pvalue=0.05442800219094721, df=4)


In [29]:
class_names = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers',
              'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']

In [30]:
report = classification_report(y_true_classes , y_pred_classes_ensemble, target_names=class_names)
print("Ablation-without WGANGP - Static Ensemble Model (5-Fold CV):UNSW-NB15-Multiclass Classification")
print(report)

Ablation static ensemble without WGAN-GP-Classification Report -  Model (5-Fold CV):UNSW-NB15-Multiclass Classification
                precision    recall  f1-score   support

      Analysis       0.24      0.76      0.37      2677
      Backdoor       0.45      0.88      0.60      2329
           DoS       0.87      0.89      0.88     16353
      Exploits       0.92      0.94      0.93     44525
       Fuzzers       0.96      0.93      0.95     24246
       Generic       0.95      0.85      0.89     58871
        Normal       0.97      0.81      0.88     93000
Reconnaissance       0.62      0.91      0.73     13987
     Shellcode       0.38      0.85      0.53      1511
         Worms       0.00      0.14      0.01       174

      accuracy                           0.86    257673
     macro avg       0.64      0.80      0.68    257673
  weighted avg       0.91      0.86      0.88    257673

